# 01 Data pipeline

Freeze the FRED DEXUSEU snapshot and build past-only features, forward variance targets, chronological splits, and manifests.

## Setup

In [1]:
# setup: clone repo, install deps, mount drive for saving models
import os, sys, subprocess
from google.colab import drive

REPO_URL = "https://github.com/BerkayPolat-lab/gru-currency-volatility-distillation.git"
REPO_DIR = "/content/gru-currency-volatility-distillation"

# clone fresh, or pull latest if already cloned this session
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
os.chdir(f"{REPO_DIR}/model")  # all paths in the config are relative to model/

sys.path.insert(0, f"{REPO_DIR}/model/src")  # lets notebooks do: from gru_vol import ...

# drive is where checkpoints live, so a colab disconnect doesn't wipe trained models
drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/gru_vol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
print("ready, working dir:", os.getcwd())

Mounted at /content/drive
ready, working dir: /content/gru-currency-volatility-distillation/model


In [2]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)
CFG

{'data': {'source': 'FRED',
  'series_id': 'DEXUSEU',
  'start': '2000-01-01',
  'end': '2025-12-31',
  'snapshot_path': 'data/raw/DEXUSEU.csv',
  'interpolate_missing': False},
 'features': {'window': 60,
  'trailing_vol_windows': [5, 20],
  'inputs': ['log_return', 'squared_return', 'vol_5', 'vol_20']},
 'target': {'horizon': 5, 'scaling': 'divide_by_train_mean'},
 'splits': {'train': ['2000-01-01', '2017-12-31'],
  'validation': ['2018-01-01', '2020-12-31'],
  'test': ['2021-01-01', '2025-12-31'],
  'drop_examples_crossing_split': True},
 'models': {'teacher': {'num_layers': 3, 'hidden_size': 128, 'dropout': 0.2},
  'student': {'num_layers': 1, 'hidden_size': 32, 'dropout': 0.0},
  'output_transform': 'softplus'},
 'historical_control': {'window': 20},
 'training': {'optimizer': 'adamw',
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'batch_size': 32,
  'max_epochs': 100,
  'early_stopping_metric': 'val_mse',
  'early_stopping_patience': 10,
  'seeds': [0, 1, 2]},
 'distillat

## Download and freeze the snapshot

Download DEXUSEU for 2000-2025, save a dated CSV, and record its hash in the manifest.

In [3]:
import pandas as pd

exchange_rate = (pd.read_csv(CFG["data"]["snapshot_path"], parse_dates=["observation_date"], index_col="observation_date")["DEXUSEU"].loc[CFG["data"]["start"]:CFG["data"]["end"]].dropna())

print(exchange_rate)

observation_date
2000-01-03    1.0155
2000-01-04    1.0309
2000-01-05    1.0335
2000-01-06    1.0324
2000-01-07    1.0294
               ...  
2025-12-24    1.1780
2025-12-26    1.1766
2025-12-29    1.1756
2025-12-30    1.1765
2025-12-31    1.1736
Name: DEXUSEU, Length: 6518, dtype: float64


## Clean and compute log returns

Keep only observed dates with no interpolation, then compute `r(t) = log(P(t) / P(t-1))`.

In [4]:
import numpy as np

log_returns = np.log(exchange_rate).diff().dropna().rename("log_return")
squared_returns = (log_returns ** 2).rename("squared_return")

In [5]:
print(f"{log_returns.name} max: {log_returns.max():.6f} on {log_returns.idxmax().date()}")
print(f"{log_returns.name} min: {log_returns.min():.6f} on {log_returns.idxmin().date()}")

log_return max: 0.046208 on 2009-03-19
log_return min: -0.030031 on 2008-12-19


In [6]:
rms = np.sqrt((squared_returns).mean())
print(f"{log_returns.name} RMS: {rms:.6f}")

log_return RMS: 0.005795


## Input features

Returns, squared returns, and trailing 5- and 20-observation volatility, all ending at the forecast origin.

In [7]:
vol_5 = np.sqrt(squared_returns.rolling(5).mean()).rename("vol_5")
vol_20 = np.sqrt(squared_returns.rolling(20).mean()).rename("vol_20")

print(vol_5.head())

observation_date
2000-01-04       NaN
2000-01-05       NaN
2000-01-06       NaN
2000-01-07       NaN
2000-01-10    0.0072
Name: vol_5, dtype: float64


In [8]:
print(vol_20.head(20))

observation_date
2000-01-04         NaN
2000-01-05         NaN
2000-01-06         NaN
2000-01-07         NaN
2000-01-10         NaN
2000-01-11         NaN
2000-01-12         NaN
2000-01-13         NaN
2000-01-14         NaN
2000-01-18         NaN
2000-01-19         NaN
2000-01-20         NaN
2000-01-21         NaN
2000-01-24         NaN
2000-01-25         NaN
2000-01-26         NaN
2000-01-27         NaN
2000-01-28         NaN
2000-01-31         NaN
2000-02-01    0.006808
Name: vol_20, dtype: float64


In [9]:
features = pd.concat([log_returns, squared_returns, vol_5, vol_20], axis=1).dropna()
print(features)

                  log_return  squared_return     vol_5    vol_20
observation_date                                                
2000-02-01         -0.002668    7.119880e-06  0.008080  0.006808
2000-02-02          0.003795    1.440256e-05  0.008147  0.005979
2000-02-03          0.012109    1.466285e-04  0.008132  0.006539
2000-02-04         -0.012928    1.671426e-04  0.008197  0.007146
2000-02-07          0.002354    5.540304e-06  0.008256  0.007135
...                      ...             ...       ...       ...
2025-12-24          0.000340    1.153392e-07  0.001685  0.002900
2025-12-26         -0.001189    1.414106e-06  0.001612  0.002818
2025-12-29         -0.000850    7.229555e-07  0.001628  0.002819
2025-12-30          0.000765    5.856437e-07  0.001243  0.002790
2025-12-31         -0.002468    6.090932e-06  0.001336  0.002796

[6498 rows x 4 columns]


## Targets

`v(t,h) = (1/h) * sum r(t+j)^2` for `j = 1..h`, with `h = 5`.

In [10]:
h = CFG["target"]["horizon"]
r2 = features["squared_return"]
rolling_mean = r2.rolling(h).mean()
target = rolling_mean.shift(-h).rename("target")

In [11]:
i = 1000
assert np.isclose(target.iloc[i], r2.iloc[i + 1 : i + 1 + h].mean())

# only the last h rows have no target
assert target.isna().sum() == h and target.tail(h).isna().all()

## Chronological splits

Train 2000-2017, validate 2018-2020, test 2021-2025. Drop examples whose forward target crosses a split boundary.

In [12]:
origin = features.index.to_series()
target_end = origin.shift(-h)

In [13]:
SPLITS = ["train", "validation", "test"]

def assign_split(dates):
    conditions = [dates.between(*CFG["splits"][name]) for name in SPLITS]
    labels = np.select(conditions, SPLITS, default="none")
    return pd.Series(labels, index=dates.index)

In [14]:
split = assign_split(origin)
target_split = assign_split(target_end)

examples = features.assign(
    target=target,
    target_end=target_end,
    split=split,
    target_split=target_split,
)

In [15]:
keep = examples["split"].eq(examples["target_split"])

In [16]:
print(examples.loc[~keep, ["split", "target_split"]].value_counts())
examples = examples[keep]
print(examples["split"].value_counts())

split       target_split
test        none            5
train       validation      5
validation  test            5
Name: count, dtype: int64
split
train         4496
test          1244
validation     743
Name: count, dtype: int64


## Scaling

z score each input window using rolling 60 day mean and std from the origin day (past only). Target is next 5 day variance divided by trailing 60 day variance (ratio)

In [17]:
# rolling 60 day z score of inputs + ratio target, past only
from numpy.lib.stride_tricks import sliding_window_view

N = 60  # scaling window, same as input window
W = CFG["features"]["window"]
inputs = CFG["features"]["inputs"]  # log_return, squared_return, vol_5, vol_20

# trailing stats per feature, each day uses only last 60 days incl itself
roll_mean = features[inputs].rolling(N).mean()
roll_std = features[inputs].rolling(N).std()
var_60 = squared_returns.rolling(N).mean().reindex(features.index).rename("var_60")  # for ratio target

# 60 day input windows of raw features, shape (n_windows, 60, 4)
windows = sliding_window_view(features[inputs].to_numpy(), window_shape=W, axis=0).transpose(0, 2, 1)

# keep examples w/ full 60 day history
row = features.index.get_indexer(examples.index)
has_history = row >= W - 1
examples_w = examples[has_history].assign(var_60=var_60.loc[examples.index[has_history]])
row = row[has_history]

# z score each window w/ stats from its origin day -> same rule for train, val, test, live deployment consistent
origin = examples_w.index
mu = roll_mean.loc[origin].to_numpy()[:, None, :]
sd = roll_std.loc[origin].to_numpy()[:, None, :]
X = ((windows[row - (W - 1)] - mu) / sd).astype(np.float32)

# ratio target: next 5 day var vs last 60 days (1 = "same as recent", i.e. HC forecast)
examples_w = examples_w.assign(target_ratio=examples_w["target"] / examples_w["var_60"])
y = examples_w["target_ratio"].to_numpy(dtype=np.float32)

print(X.shape, y.shape)
print("input mean", X.reshape(-1, 4).mean(0).round(2), "std", X.reshape(-1, 4).std(0).round(2))
print(examples_w.groupby("split")["target_ratio"].describe())

(6424, 60, 4) (6424,)
input mean [ 0. -0. -0.  0.] std [0.99 0.99 0.99 0.99]
             count      mean       std       min       25%       50%  \
split                                                                  
test        1244.0  1.086111  1.133285  0.034945  0.424827  0.763549   
train       4437.0  1.058099  0.889792  0.018882  0.501474  0.838863   
validation   743.0  1.085400  1.142229  0.066510  0.509040  0.810624   

                 75%        max  
split                            
test        1.349371  11.926988  
train       1.333965  11.027975  
validation  1.269672  10.198201  


## Save manifests

Save processed arrays, readable example table, and manifest.

In [18]:
import hashlib, json

snapshot_path = CFG["data"]["snapshot_path"]
with open(snapshot_path, "rb") as f:
    snapshot_sha256 = hashlib.sha256(f.read()).hexdigest()

raw_in_range = prices_raw = pd.read_csv(
    snapshot_path, parse_dates=["observation_date"], index_col="observation_date"
)["DEXUSEU"].loc[CFG["data"]["start"]:CFG["data"]["end"]]

In [19]:
# high vol cutoff in original units, fixed from train targets only
train_w = examples_w["split"].eq("train")
high_vol_cutoff = float(examples_w.loc[train_w, "target"].quantile(CFG["evaluation"]["high_vol_quantile"]))

# save arrays once so every notebook loads exact same data
PROCESSED = "data/processed"
os.makedirs(PROCESSED, exist_ok=True)

arrays = {}
for name in SPLITS:
    mask = examples_w["split"].eq(name).to_numpy()
    arrays[f"X_{name}"] = X[mask]  # inputs, (n, 60, 4), rolling z scored
    arrays[f"y_{name}"] = y[mask]  # ratio target, what models train on
    arrays[f"var60_{name}"] = examples_w["var_60"].to_numpy()[mask]  # to rescale preds: forecast = ratio * var_60
    arrays[f"target_{name}"] = examples_w["target"].to_numpy()[mask]  # original units, for MSE
    arrays[f"dates_{name}"] = examples_w.index[mask].strftime("%Y-%m-%d").to_numpy()
np.savez_compressed(f"{PROCESSED}/splits.npz", **arrays)

# readable table, one row per example, for baseline + plots
examples_w.drop(columns=["target_split"]).to_csv(f"{PROCESSED}/examples.csv", index_label="date")

def split_summary(name):
    rows = examples_w[examples_w["split"].eq(name)]
    return {
        "first_origin": rows.index[0].date().isoformat(),
        "last_origin": rows.index[-1].date().isoformat(),
        "last_target_day": rows["target_end"].iloc[-1].date().isoformat(),
        "examples": len(rows),
        "high_vol_examples": int((rows["target"] > high_vol_cutoff).sum()),
    }

manifest = {
    "created": pd.Timestamp.today().date().isoformat(),
    "snapshot": {
        "file": snapshot_path,
        "sha256": snapshot_sha256,
        "rows_in_range": len(raw_in_range),
        "rows_missing_dropped": int(raw_in_range.isna().sum()),
        "observed_prices": int(raw_in_range.notna().sum()),
        "first_date": raw_in_range.dropna().index[0].date().isoformat(),
        "last_date": raw_in_range.dropna().index[-1].date().isoformat(),
    },
    "examples": {
        "dropped_insufficient_history": int((~has_history).sum()),
        "dropped_crossing_boundary": int((~keep & target_split.ne("none")).sum()),
        "dropped_no_future_target": int(target_split.eq("none").sum()),
        "splits": {name: split_summary(name) for name in SPLITS},
    },
    "features": {
        "window": W,
        "inputs": inputs,
        "scaling": f"rolling {N}-obs z score per feature, stats from origin day, past only",
    },
    "target": {
        "horizon": h,
        "model_target": f"ratio: next {h}-obs mean sq return / trailing {N}-obs var",
        "high_vol_quantile": CFG["evaluation"]["high_vol_quantile"],
        "high_vol_cutoff": high_vol_cutoff,  # original units
    },
}
for folder in ["artifacts/manifests", f"{ARTIFACTS}/manifests"]:
    os.makedirs(folder, exist_ok=True)
    with open(f"{folder}/data_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

for k, v in arrays.items():
    print(k, v.shape)
print("high_vol_cutoff:", high_vol_cutoff)

X_train (4437, 60, 4)
y_train (4437,)
var60_train (4437,)
target_train (4437,)
dates_train (4437,)
X_validation (743, 60, 4)
y_validation (743,)
var60_validation (743,)
target_validation (743,)
dates_validation (743,)
X_test (1244, 60, 4)
y_test (1244,)
var60_test (1244,)
target_test (1244,)
dates_test (1244,)
high_vol_cutoff: 8.035056898160317e-05
